# Phase 1 Comprehensive Results: Scaled Self-Verification Baseline

**Student:** Tulasi Padamata  
**Project:** Multi-Agent Hallucination Detection in RAG Systems  
**Date:** March 12, 2026

This notebook presents the complete Phase 1 results with:
- Scaled self-verification baseline (600 samples)
- Statistical validation (bootstrap confidence intervals)
- Comprehensive metrics using unified evaluation module
- Clear separation of Track A (hallucination detection) vs Track B (RAG QA)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from evaluation.metrics import (
    case_level_metrics,
    per_task_metrics,
    compute_span_level_metrics,
    bootstrap_confidence_interval,
    compute_confusion_matrix,
    compute_classification_report,
    hallucination_type_breakdown,
    full_evaluation_report
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## Track A: Hallucination Detection Evaluation

This track evaluates methods that detect hallucinations in LLM-generated responses.

**Methods:**
1. RAGTruth LLaMA-2-13B (official baseline, supervised fine-tuning)
2. Self-Verification GPT-4o-mini (our baseline, zero-shot inference)
3. Multi-Agent Pipeline (proposed, CKPT3)

**Evaluation Protocol:**
- Test set: RAGTruth test split (balanced across QA, Summary, Data2txt)
- Ground truth: Human-annotated hallucination spans
- Metrics: Case-level F1, Span-level F1, Precision, Recall
- Statistical validation: 95% bootstrap confidence intervals

In [ ]:
def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            records.append(json.loads(line))
    return records

predictions_path = '../results/gpt_baseline_predictions.jsonl'
records = load_jsonl(predictions_path)
print(f"Loaded {len(records)} predictions")

task_counts = pd.Series([r['task_type'] for r in records]).value_counts()
print(f"\nTask distribution:")
print(task_counts)

### 1. Case-Level Metrics (Primary)

In [ ]:
case_metrics = case_level_metrics(records)

print("Overall Case-Level Performance:")
print(f"  Precision: {case_metrics['precision']:.4f}")
print(f"  Recall:    {case_metrics['recall']:.4f}")
print(f"  F1 Score:  {case_metrics['f1']:.4f}")
print(f"  Samples:   {case_metrics['n_samples']}")

### 2. Statistical Validation: Bootstrap Confidence Intervals

In [ ]:
bootstrap_ci = bootstrap_confidence_interval(records, n_iterations=1000, confidence=0.95)

print("95% Bootstrap Confidence Interval:")
print(f"  Mean F1:   {bootstrap_ci['mean_f1']:.4f}")
print(f"  CI:        [{bootstrap_ci['ci_lower']:.4f}, {bootstrap_ci['ci_upper']:.4f}]")
print(f"  Width:     {bootstrap_ci['ci_upper'] - bootstrap_ci['ci_lower']:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar([0], [bootstrap_ci['mean_f1']], 
            yerr=[[bootstrap_ci['mean_f1'] - bootstrap_ci['ci_lower']], 
                  [bootstrap_ci['ci_upper'] - bootstrap_ci['mean_f1']]],
            fmt='o', markersize=10, capsize=10, capthick=2, linewidth=2)
ax.set_xlim(-0.5, 0.5)
ax.set_ylim(0, 1)
ax.set_ylabel('F1 Score')
ax.set_title('Self-Verification Baseline: F1 with 95% CI')
ax.set_xticks([0])
ax.set_xticklabels(['GPT-4o-mini\nSelf-Verification'])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3. Per-Task Breakdown

In [ ]:
task_metrics = per_task_metrics(records)

print("Per-Task Performance:")
for task, metrics in task_metrics.items():
    print(f"\n{task}:")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1 Score:  {metrics['f1']:.4f}")
    print(f"  Samples:   {metrics['n_samples']}")

task_df = pd.DataFrame([
    {'Task': task, 'Metric': 'Precision', 'Score': m['precision']} for task, m in task_metrics.items()
] + [
    {'Task': task, 'Metric': 'Recall', 'Score': m['recall']} for task, m in task_metrics.items()
] + [
    {'Task': task, 'Metric': 'F1', 'Score': m['f1']} for task, m in task_metrics.items()
])

fig, ax = plt.subplots(figsize=(10, 6))
task_pivot = task_df.pivot(index='Task', columns='Metric', values='Score')
task_pivot[['Precision', 'Recall', 'F1']].plot(kind='bar', ax=ax)
ax.set_ylabel('Score')
ax.set_title('Self-Verification Performance by Task Type')
ax.set_ylim(0, 1)
ax.legend(title='Metric')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 4. Span-Level Metrics

In [ ]:
span_metrics = compute_span_level_metrics(records)

print("Span-Level Performance:")
print(f"  Mean F1:   {span_metrics['mean_span_f1']:.4f}")
print(f"  Median F1: {span_metrics['median_span_f1']:.4f}")
print(f"  Std F1:    {span_metrics['std_span_f1']:.4f}")

span_f1_per_task = {}
for task in ['QA', 'Summary', 'Data2txt']:
    task_records = [r for r in records if r['task_type'] == task]
    task_span = compute_span_level_metrics(task_records)
    span_f1_per_task[task] = task_span['mean_span_f1']
    print(f"\n{task} Span-Level F1: {task_span['mean_span_f1']:.4f}")

### 5. Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = compute_confusion_matrix(records)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, 
                               display_labels=['No Hallucination', 'Hallucination'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Self-Verification Confusion Matrix')
plt.tight_layout()
plt.show()

accuracy = (cm[0][0] + cm[1][1]) / cm.sum()
print(f"\nAccuracy: {accuracy:.4f}")
print(f"True Negatives:  {cm[0][0]}")
print(f"False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}")
print(f"True Positives:  {cm[1][1]}")

### 6. Classification Report

In [ ]:
class_report = compute_classification_report(records)
print(class_report)

### 7. Hallucination Type Breakdown

In [ ]:
type_breakdown = hallucination_type_breakdown(records)

print("Hallucination Types in Test Set:")
for htype, count in type_breakdown.items():
    print(f"  {htype:<25} {count}")

fig, ax = plt.subplots(figsize=(10, 6))
types = list(type_breakdown.keys())
counts = list(type_breakdown.values())
ax.bar(types, counts)
ax.set_ylabel('Count')
ax.set_title('Hallucination Type Distribution in Test Set')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---

## Track B: RAG QA Performance (Supplementary)

This track evaluates the RAG system's ability to generate answers from retrieved passages.

**Method:** GPT-4o-mini RAG Generator

**Evaluation Protocol:**
- Test set: 100 QA samples from RAGTruth
- Ground truth: None (RAGTruth does not provide gold answers)
- Metrics: Answerable rate, answer length distribution

**Note:** This is NOT directly comparable to Track A methods because it solves a different task (answer generation vs hallucination detection).

In [ ]:
rag_answers_path = '../baselines/rag_baseline_answers.jsonl'
rag_records = load_jsonl(rag_answers_path)
print(f"Loaded {len(rag_records)} RAG QA answers")

answerable = sum(1 for r in rag_records if 'cannot answer' not in r['generated_answer'].lower())
answerable_rate = answerable / len(rag_records)

print(f"\nAnswerable Rate: {answerable_rate:.2%}")
print(f"Answerable: {answerable}/{len(rag_records)}")

word_counts = [len(r['generated_answer'].split()) for r in rag_records]
print(f"\nAnswer Length Statistics:")
print(f"  Mean:   {np.mean(word_counts):.1f} words")
print(f"  Median: {np.median(word_counts):.1f} words")
print(f"  Min:    {np.min(word_counts)} words")
print(f"  Max:    {np.max(word_counts)} words")

---

## Summary: Track A vs Track B

### Track A: Hallucination Detection
- **Task:** Binary classification (hallucinated vs not hallucinated)
- **Ground Truth:** Human annotations
- **Primary Metric:** Case-level F1
- **Self-Verification Baseline F1:** [value from above]
- **Next:** Multi-agent pipeline (CKPT3) will be evaluated using the same protocol

### Track B: RAG QA Generation
- **Task:** Generate answers from passages
- **Ground Truth:** None
- **Primary Metric:** Answerable rate
- **RAG System Answerable Rate:** [value from above]
- **Purpose:** Demonstrates end-to-end RAG pipeline works (per professor feedback)

**These two tracks are NOT directly comparable because they solve different tasks.**

## References

1. Wu et al. (2023). RAGTruth: A Hallucination Corpus for Developing Trustworthy Retrieval-Augmented Language Models. arXiv:2401.00396
2. Manakul et al. (2023). SelfCheckGPT: Zero-Resource Black-Box Hallucination Detection for Generative Large Language Models. arXiv:2303.08896